# 1. Max 3-Cut and the two Hamiltonians

Max 3-Cut asks you to paint the vertices of a graph with three colours so that
as many edges as possible have differently coloured ends. It is the natural
problem for a qutrit: one vertex, one qutrit, three colours.

This notebook builds the two Hamiltonians the algorithm needs and checks them
against a brute-force solution. Nothing here is variational yet.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np

import qutip as qt
from funciones.utilidades import Hp_qutip, Hi_qutip, Jz1

# G_1 of the paper: six vertices, ten edges. Vertices are 1-indexed.
n = 6
edges = [(1, 2), (1, 3), (1, 4), (1, 5), (2, 4),
         (2, 6), (3, 4), (3, 6), (4, 5), (5, 6)]

print(f"{n} qutrits, {len(edges)} edges -> Hilbert space of dimension {3**n}")

6 qutrits, 10 edges -> Hilbert space of dimension 729


## The ternary variable

A qutrit level is a colour. The paper uses $L_z$ to label them, because it is
diagonal in the computational basis and its three eigenvalues are exactly the
three values a ternary variable can take:

$$v_i \in \{-1, 0, 1\}.$$

In [2]:
print("L_z =")
print(np.real(Jz1.full()))
print("\neigenvalues:", np.real(Jz1.eigenenergies()))

L_z =
[[ 1.  0.  0.]
 [ 0.  0.  0.]
 [ 0.  0. -1.]]

eigenvalues: [-1.  0.  1.]


## The cost Hamiltonian

Per edge $(i,j)$ the paper writes

$$H_C\big|_{(i,j)} = L_z^{(i)}L_z^{(j)} - 2\left[(L_z^{(i)})^2 + (L_z^{(j)})^2\right] + 3(L_z^{(i)})^2(L_z^{(j)})^2 .$$

It is diagonal, so its ground state *is* a colouring. Let us check the one
thing that matters: that the term is lower when the two ends differ.

In [3]:
import itertools

Hc = Hp_qutip(n, edges)
diag = np.real(Hc.diag())

# Un solo par de vertices, para ver que hace el termino de arista.
par = Hp_qutip(2, [(1, 2)])
print("cost of one edge, by colouring of its two ends:")
for a, b in itertools.product(range(3), repeat=2):
    idx = a * 3 + b
    print(f"  ({a},{b}) -> {np.real(par.diag()[idx]):+.0f}"
          f"   {'cut' if a != b else 'not cut'}")

cost of one edge, by colouring of its two ends:
  (0,0) -> +0   not cut
  (0,1) -> -2   cut
  (0,2) -> -2   cut
  (1,0) -> -2   cut
  (1,1) -> +0   not cut
  (1,2) -> -2   cut
  (2,0) -> -2   cut
  (2,1) -> -2   cut
  (2,2) -> +0   not cut


A cut edge contributes $-2$ and an uncut one $0$. So the ground energy counts
cut edges: $|E_0|/2$ of them.

In [4]:
E0 = float(np.min(diag))
print(f"E_0 = {E0:.0f}   ->   {abs(E0)/2:.0f} cut edges out of {len(edges)}")
print(f"ground-state degeneracy: {int(np.sum(np.isclose(diag, E0)))}")

E_0 = -20   ->   10 cut edges out of 10
ground-state degeneracy: 12


## Brute force, to be sure

$3^6 = 729$ colourings is nothing. Let us enumerate them and confirm the
spectrum is telling the truth.

In [5]:
mejor, optimos = -1, []
for col in itertools.product(range(3), repeat=n):
    cortadas = sum(1 for i, j in edges if col[i-1] != col[j-1])
    if cortadas > mejor:
        mejor, optimos = cortadas, [col]
    elif cortadas == mejor:
        optimos.append(col)

print(f"brute force: {mejor}/{len(edges)} edges cut, by {len(optimos)} colourings")
print(f"spectrum   : {abs(E0)/2:.0f} edges, degeneracy {int(np.sum(np.isclose(diag, E0)))}")
print("\nagree:", mejor == abs(E0)/2 and len(optimos) == int(np.sum(np.isclose(diag, E0))))
print("\none optimal colouring:", optimos[0])

brute force: 10/10 edges cut, by 12 colourings
spectrum   : 10 edges, degeneracy 12

agree: True

one optimal colouring: (0, 1, 1, 2, 1, 0)


All ten edges are cut, because this graph happens to be 3-colourable.

## The mixer

$$H_M = -\sum_j\left[\sqrt2\,L_x^{(j)} + (L_z^{(j)})^2\right]$$

That specific combination is chosen so its ground state is the uniform
superposition $|{+_3}\rangle^{\otimes n}$, which is the reference state ADAPT
starts from. Let us verify that.

In [6]:
Hm = Hi_qutip(n, omega0=1)
vals, vecs = Hm.eigenstates()
phi_g = vecs[0]

# El estado de referencia del algoritmo: superposicion uniforme de las 3^n
# coloraciones. En qutip 5 el producto interno ya devuelve un escalar.
uniforme = qt.tensor([qt.Qobj(np.ones((3, 1)) / np.sqrt(3))] * n)
solape = abs(uniforme.overlap(phi_g))

print(f"ground energy of H_M is non-degenerate: "
      f"{not np.isclose(vals[0], vals[1])}")
print(f"|<uniform|ground state of H_M>| = {solape:.12f}")
print(f"<phi_g|H_C|phi_g> = {float(np.real(qt.expect(Hc, phi_g))):.4f}"
      f"   (vs E_0 = {E0:.0f})")

ground energy of H_M is non-degenerate: True
|<uniform|ground state of H_M>| = 1.000000000000
<phi_g|H_C|phi_g> = -13.3333   (vs E_0 = -20)


The reference state is exactly the uniform superposition, and it starts at
$\langle H_C\rangle = -40/3 \approx -13.33$ against a ground energy of $-20$.
That is the honest starting point: the state is spread evenly over all $729$
colourings, so it already "contains" the answer with amplitude $3^{-3}$ — the
whole job of the algorithm is to concentrate it.

In the notation of the paper this is $\epsilon_{\rm rel} = 1/3$, which is where
every curve in Figs. 1-3 begins.